# 随机森林对照实验（RF_Funning）

使用与 LoRA 实验一致的数据划分与超参数，输出统一指标格式。

In [1]:
"""
第一部分：导入依赖
"""

import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, log_loss

from Bert_Config import CONFIG, setup_seed, load_raw_data, split_data

print("✅ 所有库导入完成")


"""
第二部分：统一配置
"""

MODEL_NAME = "RandomForest"
DATA_PATH = CONFIG["DATA_DIR"]
RANDOM_SEED = CONFIG["RANDOM_SEED"]

MAX_FEATURES = CONFIG["CLASSIC_MAX_FEATURES"]
NGRAM_RANGE = CONFIG["CLASSIC_NGRAM_RANGE"]
MIN_DF = CONFIG["CLASSIC_MIN_DF"]
MAX_DF = CONFIG["CLASSIC_MAX_DF"]

print("✅ 配置加载完成")


"""
第三部分：定义工具函数
"""


def build_vectorizer():
    return TfidfVectorizer(
        analyzer="char",
        ngram_range=NGRAM_RANGE,
        max_features=MAX_FEATURES,
        min_df=MIN_DF,
        max_df=MAX_DF,
    )


def evaluate(y_true, y_pred, y_prob=None):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    loss = None
    if y_prob is not None:
        loss = log_loss(y_true, y_prob)
    return acc, precision, recall, f1, loss


def print_report(train_metrics, val_metrics, test_metrics, elapsed_sec):
    elapsed_min = elapsed_sec / 60
    print("\n" + "=" * 50)
    print(f"模型: {MODEL_NAME}")
    print(
        f"训练集 - Acc: {train_metrics[0]:.3f} | Precision: {train_metrics[1]:.3f} | "
        f"Recall: {train_metrics[2]:.3f} | F1: {train_metrics[3]:.3f} | "
        f"Loss: {train_metrics[4]:.3f}"
    )
    print(
        f"验证集 - Acc: {val_metrics[0]:.3f} | Precision: {val_metrics[1]:.3f} | "
        f"Recall: {val_metrics[2]:.3f} | F1: {val_metrics[3]:.3f} | "
        f"Loss: {val_metrics[4]:.3f}"
    )
    print(
        f"测试集 - Acc: {test_metrics[0]:.3f} | Precision: {test_metrics[1]:.3f} | "
        f"Recall: {test_metrics[2]:.3f} | F1: {test_metrics[3]:.3f} | "
        f"Loss: {test_metrics[4]:.3f}"
    )
    print(f"训练时间: {elapsed_sec:.1f} 秒 ({elapsed_min:.2f} 分钟)")
    print("=" * 50)

print("✅ 工具函数定义完成")


"""
第四部分：准备数据
"""

setup_seed(RANDOM_SEED)
df = load_raw_data(DATA_PATH)
train_df, val_df, test_df = split_data(df, RANDOM_SEED)

vectorizer = build_vectorizer()
x_train = vectorizer.fit_transform(train_df["review"])
x_val = vectorizer.transform(val_df["review"])
x_test = vectorizer.transform(test_df["review"])

y_train = train_df["label"].astype(int).values
y_val = val_df["label"].astype(int).values
y_test = test_df["label"].astype(int).values

print("✅ 数据准备完成")


"""
第五部分：训练与评估
"""

model = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

start = time.time()
model.fit(x_train, y_train)
elapsed = time.time() - start

train_pred = model.predict(x_train)
val_pred = model.predict(x_val)
test_pred = model.predict(x_test)

train_prob = model.predict_proba(x_train)
val_prob = model.predict_proba(x_val)
test_prob = model.predict_proba(x_test)

train_metrics = evaluate(y_train, train_pred, train_prob)
val_metrics = evaluate(y_val, val_pred, val_prob)
test_metrics = evaluate(y_test, test_pred, test_prob)

print_report(train_metrics, val_metrics, test_metrics, elapsed)


C:\Users\19836\miniconda3\envs\py8\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 所有库导入完成
✅ 配置加载完成
✅ 工具函数定义完成
✅ 数据准备完成

模型: RandomForest
训练集 - Acc: 1.000 | Precision: 1.000 | Recall: 0.999 | F1: 1.000 | Loss: 0.086
验证集 - Acc: 0.880 | Precision: 0.879 | Recall: 0.743 | F1: 0.805 | Loss: 0.314
测试集 - Acc: 0.874 | Precision: 0.893 | Recall: 0.708 | F1: 0.789 | Loss: 0.321
训练时间: 2.6 秒 (0.04 分钟)
